In [1]:
!pip install langchain-huggingface faiss-cpu langchain-groq -q
!pip install faiss-cpu langchain-community -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 61.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 23.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 40.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 4.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [2]:
import os
import pandas as pd
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

os.environ['GROQ_API_KEY'] = 'your-key-here'

# ── Step 1: Load your real London property CSV ───────────────
df = pd.read_csv('/content/sample_data/kaggle_london_house_price_data.csv')

# Clean — keep rows with price and area
df = df[df['saleEstimate_currentPrice'].notna()]
df = df[df['saleEstimate_currentPrice'] > 0]
df = df.head(50)  # first 50 properties
print(f"✓ Loaded {len(df)} properties")

# ── Step 2: Convert each row to a LangChain Document ────────
documents = []
for _, row in df.iterrows():
    price    = row['saleEstimate_currentPrice']
    area     = row.get('outcode', 'Unknown')
    ptype    = row.get('propertyType', 'Unknown')
    sqm      = row.get('floorAreaSqM', 'Unknown')

    content = (
        f"Property in {area}. "
        f"Type: {ptype}. "
        f"Price: £{price:,.0f}. "
        f"Floor area: {sqm} sqm."
    )

    documents.append(Document(
        page_content=content,
        metadata={
            "area":  area,
            "price": price,
            "type":  str(ptype)
        }
    ))

print(f"✓ {len(documents)} documents created")

# ── Step 3: Build vector store ───────────────────────────────
embeddings  = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(documents, embeddings)
retriever   = vectorstore.as_retriever(search_kwargs={"k": 5})
print("✓ Vector store built")

# ── Step 4: RAG chain ────────────────────────────────────────
model  = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)
parser = StrOutputParser()

rag_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a London property investment advisor.
Answer using ONLY the property listings provided below.
Be specific — mention prices and locations from the data.
If the answer is not in the data, say 'I don't have that information.'
Never make up property details.

Listings:
{context}"""),
    ("human", "{question}")
])

def format_docs(docs):
    return "\n".join([
        f"- [{doc.metadata.get('area','Unknown')}]: {doc.page_content}"
        for doc in docs
    ])

rag_chain = (
    {"context":  retriever | format_docs,
     "question": RunnablePassthrough()}
    | rag_prompt | model | parser
)

print("✓ RAG chain ready")

# ── Step 5: Ask 5 investor questions ────────────────────────
questions = [
    "What types of properties are available?",
    "Which area has the most listings?",
    "What is the most expensive property and where is it?",
    "What is the cheapest property available?",
    "Which properties are best value based on price and size?"
]

print("\n" + "="*55)
print("London Property Q&A Assistant")
print("="*55)

for q in questions:
    print(f"\nQ: {q}")
    print(f"A: {rag_chain.invoke(q)}")

# ── Step 6: Stretch — save vector store to disk ──────────────
vectorstore.save_local("property_index")
print("\n✓ Vector store saved to property_index/")
print("  Load next time with:")
print("  vectorstore = FAISS.load_local('property_index', embeddings, allow_dangerous_deserialization=True)")

/tmp/ipykernel_4406/2279034199.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


✓ Loaded 50 properties
✓ 50 documents created


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✓ Vector store built
✓ RAG chain ready

London Property Q&A Assistant

Q: What types of properties are available?
A: The available properties are: 
1. Purpose Built Flat (available in N6, NW4, and W1W)
2. Semi-Detached House (available in W7)

Q: Which area has the most listings?
A: The area SE9 has the most listings, with 2 properties: a Mid Terrace House priced at £537,000 and a Semi-Detached House priced at £601,000.

Q: What is the most expensive property and where is it?
A: The most expensive property is a Purpose Built Flat in SW10, priced at £1,388,000, with a floor area of 137.0 sqm.

Q: What is the cheapest property available?
A: The cheapest property available is in SE5, a Flat/Maisonette, priced at £388,000, with a floor area of 64.0 sqm.

Q: Which properties are best value based on price and size?
A: To determine the best value based on price and size, we need to calculate the price per square meter for each property.

- [W1W]: £477,000 / 40.0 sqm = £11,925 per sqm
- [W1W]: